```
token_ids (x, y)
    ↓
TransformerLM(x)
    ↓
logits [B, S, V]
    ↓
CrossEntropy(logits, y)
    ↓
scalar loss
    ↓
loss.backward()
    ↓
所有参数得到 grad
    ↓
gradient clipping（可选）
    ↓
AdamW.step()
    ↓
参数更新
    ↓
scheduler 改下一个 step 的 lr
```

## 5.1 损失函数：交叉熵
模型说“我觉得下一个 token 是这个”，
真实答案说“不是，是那个”，
那这个“差距”到底怎么量化？

前面堆叠Transformer Blocks得到的Logits表示：

**对于 batch 中每个样本、每个位置，模型都给词表里的每个 token 一个分数**

这个分数还不是概率，只是 logits。

与此同时，真实标签 target 的形状是`[B, S]`，里面每个位置存的是一个整数 ID，表示这个位置真正应该预测到的 token 是词表里的第几个。

交叉熵做的事非常朴素：
**把正确 token 的 logit 拉高，把错误 token 的 logit 压低。**

### 5.1.1 你到底在预测谁
模型输入如果是`x = [t0, t1, t2, t3]`，那那对应的训练目标不是“再输出一遍自己”，而是`y = [t1, t2, t3, t4]`

也就是说：
- 位置 0 用 t0 预测 t1
- 位置 1 用 t1 预测 t2
- 位置 2 用 t2 预测 t3

所以 language modeling 的训练其实天然带有一个 **shift by one**。

方案就是，在模型输出完整 logits 后，在 loss 前自己切：
- `logits[:, :-1, :]`
- `targets[:, 1:]`

### 5.1.2 从概率定义到工程公式
对于 logits 向量 $\boldsymbol{o} = [o_1, o_2, \dots, o_V]$（$V$ 是词表大小），Softmax 定义为：
$$
\text{softmax}(\boldsymbol{o})_i = \frac{e^{o_i}}{\sum_{j=1}^V e^{o_j}}
$$
取 target 对应位置的概率 $p_y = \text{softmax}(\boldsymbol{o})_y$，再计算负对数似然（NLL）：
$$
\text{Loss} = -\log(p_y) = \log\left(\sum_{j=1}^V e^{o_j}\right) - o_y
$$

工程实现里真正会写的版本是上面这种 **LogSumExp - 正确位置 logit**，而不是“先 softmax，再取正确位置，再取 log”。

因为如果直接取Softmax的话，在 LLM 训练里，logits 一旦大起来：
- **上溢**：$e^{100}$ 远超 float32 的最大值（约 $e^{88} \approx 10^{38}$），直接变成 `inf`。
- **下溢**：$e^{-100}$ 会被 float32 舍入为 `0`，导致 $\log(0)$ 变成 `-inf`。

我们想计算的核心项是：
$$
\text{LSE}(\boldsymbol{o}) = \log\left(\sum_{j=1}^V e^{o_j}\right)
$$
**引入最大值平移项**：令 $M = \max(o_1, o_2, \dots, o_V)$，则：
$$
\begin{align*}
\text{LSE}(\boldsymbol{o}) &= \log\left(\sum_{j=1}^V e^{o_j}\right) \\
&= \log\left(\sum_{j=1}^V e^{M} \cdot e^{o_j - M}\right) \\
&= \log\left(e^{M} \cdot \sum_{j=1}^V e^{o_j - M}\right) \\
&= M + \log\left(\sum_{j=1}^V e^{o_j - M}\right)
\end{align*}
$$

1.  **M 不改变数学结果**（最后加回去了）。
2.  **它只改变计算路径**：
    *   把所有数**拉回到以 0 为中心的安全区**。
    *   保证 $e^{(\cdot)}$ 最大是 $e^0 = 1$（防上溢）。
    *   保证求和时至少有一个 1（防下溢成 0）。

In [2]:
# 平时的写法
# torch.nn.functional.cross_entropy(...)

import torch
def cross_entropy(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    """
    数值稳定的 token-level cross entropy。

    参数:
        logits:  [B, S, V]
        targets: [B, S]

    返回:
        标量 loss
    """
    m = torch.max(logits, dim=-1, keepdim=True).values    # dim=-1取最后一个维度上的最大值，这里keepdim=True是为了保留V维度，此时形状由[B, S, V] -> [B, S, 1]

    target_logits = torch.gather(
        logits,
        dim=-1,
        index=targets.unsqueeze(-1)    # [B, S] -> [B, S, 1]
    ).squeeze(-1)    # [B, S, 1] -> [B, S]

    shifted_logits = logits - m    # [B, S, V]
    log_sum_exp = m.squeeze(-1) + torch.log(
        torch.sum(torch.exp(shifted_logits), dim=-1)    # [B, S]
    )

    # 也可以这样，Pytorch自己给的logsumexp
    # lse = torch.logsumexp(logits, dim=-1)  # [B, S]
    # return (lse - target_logits).mean()

    loss = log_sum_exp - target_logits    # [B, S]

    # batch 内的 token 平均 loss
    return loss.mean()    # 标量


> 语言模型训练里，`x` 和 `y` 是 **错一位对齐** 的

| 核心操作 | 通俗类比 | 核心数据变化 | 操作前（形状+张量内容） | 操作后（形状+张量内容） | 典型使用场景 |
|----------|----------|--------------|--------------------------|--------------------------|--------------|
| `torch.unsqueeze(dim=xxx)`<br>（必须指定维度位置） | 给张量**穿一层外套（加一层括号）**，不改动内部数据 | **完全不变**，仅新增一个长度固定为1的维度，不修改、不复制、不新增任何原有数值 | 形状：`torch.Size([2, 3])`<br>张量内容：<br>```tensor([[1, 3, 5],<br>        [2, 4, 6]])``` | 形状：`torch.Size([2, 3, 1])`<br>张量内容：<br>```tensor([[[1],<br>         [3],<br>         [5]],<br>        [[2],<br>         [4],<br>         [6]]])``` | 为`gather`索引操作匹配维度、为张量广播运算对齐形状（比如交叉熵损失中对targets标签的维度处理） |
| `torch.squeeze(dim=xxx)`<br>（不指定dim则移除所有长度为1的维度） | 给张量**脱一层多余的外套（去掉一层冗余括号）**，不改动内部数据 | **完全不变**，仅移除长度为1的维度，不修改、不丢失任何原有数值 | 形状：`torch.Size([2, 1, 3])`<br>张量内容：<br>```tensor([[[10, 20, 30]],<br>        [[40, 50, 60]]])``` | 形状：`torch.Size([2, 3])`<br>张量内容：<br>```tensor([[10, 20, 30],<br>        [40, 50, 60]])``` | 移除`keepdim=True`留下的冗余维度、还原张量的业务维度（比如交叉熵损失中对gather结果的维度还原） |



## 5.2 AdamW 优化器
loss 会给出“错多少”，但模型参数要怎么改？

训练时我们的目标是最小化：

$$
L(\theta)
$$

梯度告诉我们：

$$
g_t = \nabla_\theta L(\theta_t)
$$

**只告诉了「当前位置往哪个方向走，海拔下降最快」**，除此之外啥也不管。

而优化器 = **下山策略**，专门解决梯度管不了的问题：
  1.  要不要顺着之前跑的惯性继续走？
  2.  不同的路况，步长该不该不一样？
  3.  要不要加个稳定的刹车，防止冲过头摔下山（过拟合）？

### 5.2.1 SGD
最基础的更新方式：
$$
\theta_{t+1} = \theta_t - \eta g_t
$$
- `η`：学习率（固定的基础步长）
- `g_t`：当前时刻的梯度（指南针指的方向）

* 梯度方向一抖，它就跟着乱抖
* 遇到狭长山谷，容易左右横跳
* 遇到平坦区域，又容易走不动

### 5.2.2 Momentum
$$
m_t = \beta_1 m_{t-1} + (1-\beta_1)g_t
$$

- `m_t`：动量（当前的滑行速度）
- `β_1`：惯性系数（常用0.9，意思是「上一步的速度保留90%，当前梯度只占10%的权重」）

$$
\theta_{t+1} = \theta_t - \eta m_t
$$

* 主方向上的梯度会不断累积
* 震荡方向上的梯度会互相抵消

### 5.2.3 RMSProp
Momentum 解决的是“方向更稳”，但还没解决“步长应该自适应”。
RMSProp 的核心想法是：
- 梯度波动大的参数，步子小一点；
- 梯度一直很小的参数，步子可以大一点。

$$
v_t = \beta_2 v_{t-1} + (1-\beta_2)g_t^2
$$
- `v_t`：梯度平方的滑动平均（记录路面的颠簸程度）
- `β_2`：衰减系数（常用0.999）
- `ϵ`：极小值（防止分母为0）
$$
\theta_{t+1} = \theta_t - \frac{\eta}{\sqrt{v_t}+\epsilon} g_t
$$
也就是说，分母会根据历史梯度平方均值自动调步长：
**坑坑洼洼的路迈小步，平缓稳定的路迈大步**：
- 如果某个参数的梯度波动极大（路面颠簸，一会陡一会平），`v_t`就会很大，分母变大，步长自动缩小，防止踩空。
- 如果某个参数的梯度一直很小、很稳定（路面平缓），`v_t`就会很小，分母变小，步长自动放大，不会卡在原地。
> ！只解决了步长自适应，没保留Momentum的惯性稳方向的能力。

### 5.2.4 Adam
Adam 可以看作既要Momentum的「方向稳、带惯性」，又要RMSProp的「步长自适应」。
1.  继承Momentum：一阶矩`m_t`（管方向和惯性）
    $$m_t = \beta_1 m_{t-1} + (1-\beta_1)g_t$$
2.  继承RMSProp：二阶矩`v_t`（管自适应步长）
    $$v_t = \beta_2 v_{t-1} + (1-\beta_2)g_t^2$$
3.  偏差修正（解决初始值为0的偏差）
    $$\hat{m}_t = \frac{m_t}{1-\beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1-\beta_2^t}$$
4.  最终参数更新
    $$\theta_{t+1} = \theta_t - \eta \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}$$

`m_t`和`v_t`初始值是0，前几步的累积值会严重偏小，就像滑板刚起步，速度还没起来。除以`1-β^t`就是给前几步的数值「补加速」，迭代步数`t`越大，`β^t`越接近0，修正项就越接近1，后期就不再起作用了。

> 在「权重衰减（防止过拟合）」这件事上，处理得不干净，有耦合问题。

### 5.2.5 Weight decay
Adam 看起来已经挺完美了，但它在 **weight decay** 这件事上，处理得并不“干净”。
权重衰减的核心目的是，**每更新一步参数，就把参数往0的方向拉一点点**，防止参数变得太大，模型学死在训练集里（过拟合）。
在 SGD 里，如果你给 loss 加一个 L2 项（给大参数惩罚）：
$$L_{总}(\theta) = L(\theta) + \frac{\lambda}{2}||\theta||^2$$

| 符号 | 含义 | 核心作用 |
|------|------|----------|
| $L(\theta)$ | 原始任务损失（如交叉熵） | 管模型「任务做的准不准」 |
| $\frac{\lambda}{2} \left\|\theta\right\|_2^2$ | L2正则项（惩罚项） | 管模型「参数够不够稳、够不够平滑」 |
| $\lambda$（lambda） | 正则强度超参数（手动设置） | 控制惩罚力度：<br>- $\lambda=0$：完全不加正则<br>- $\lambda$越大：对大参数的惩罚越狠<br>- $\lambda$过大会导致模型学不到规律，出现欠拟合 |
| $\left\|\theta\right\|_2^2$ | 参数的L2范数的平方 | 通俗说就是**所有参数的平方和**。比如参数 $\theta=[w_1,w_2,w_3]$，则 $\left\|\theta\right\|_2^2 = w_1^2 + w_2^2 + w_3^2$ |
| $\frac{1}{2}$ | 固定系数 | 纯数学便利：求导时平方的导数是2倍，$\frac{1}{2}$ 刚好抵消，让求导后的式子更简洁，不影响最终效果 |
>     不加L2正则时，模型可能会把某几个参数拉得极大，完全靠这几个特征做预测，一旦测试集里这几个特征带噪声，模型直接失效；加了L2正则后，模型会把权重分散到更多特征上，不会让某几个参数「独大」，就像做决策会综合多方信息，而不是只听一个人的意见，泛化能力自然更强。
求梯度后，梯度变成：
$$g_{t总} = g_t + \lambda \theta_t$$
代入SGD更新公式：
$$\theta_{t+1} = \theta_t - \eta(g_t + \lambda \theta_t) = \underbrace{\theta_t - \eta g_t}_{正常SGD更新} - \underbrace{\eta \lambda \theta_t}_{独立的参数收缩项}$$

展开后会发现它就等价于：
* 普通梯度更新
* 再加一个把参数往 0 拉的小收缩项
所以在 SGD 语境里，大家经常把 L2 regularization 和 weight decay 混着说。

但在 Adam 里，它们不再等价 —— 问题出在 Adam 的更新不是直接减梯度，而是还要除以 $$\sqrt{\hat v_t} + \epsilon$$

如果你把 L2 项直接加进梯度，那么这部分“衰减项”也会被二阶矩缩放。也就是说，实际衰减力度会受梯度统计量影响。

weight decay 的本意应该是 **无论当前梯度长什么样，我都要稳定地把参数往 0 拉一点。**而不是“梯度大就少衰减，梯度小就多衰减”。这就是耦合问题。

AdamW 的思想就是先按 Adam 算正常梯度步长，再独立执行参数衰减。
$$\theta_{t+1} = \underbrace{\theta_t - \eta \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}}_{纯纯的Adam梯度更新，和衰减无关} - \underbrace{\eta \lambda \theta_t}_{纯纯的独立权重衰减，和梯度、二阶矩完全无关}$$

这样 weight decay 就不再被二阶矩“污染”。
#### 5.2.4.1 重要性
- 1）抑制参数无节制长大
  - 参数太大，会让激活和更新都更暴躁。尤其深层模型里，这会一点点积累成数值不稳。
- 2）让模型更偏向“简单函数”
  - 经典正则化直觉。
  - 如果两个解都能解释数据，权重更小的那个通常泛化更稳。
#### 5.2.4.2 例外
通常来说，以下参数往往 **不做 weight decay**：

* bias
* norm 层参数（`LayerNorm.weight`, `RMSNorm.weight`）
* 有时 embedding 也会特殊处理

因为这些参数更多是“尺度和偏移控制器”，不是你想强行拉向 0 的那类大矩阵权重。
> 后面训练脚本里最好做 **parameter grouping**：
> * decay group
> * no_decay group


| 参数名 | 类型 | 默认值 | 数学符号 / 对应公式 | 通俗解释 / 核心作用 | 代码中的校验规则 |
|--------|------|--------|----------------------|----------------------|------------------|
| `params` | `Iterable[Parameter]` | （无默认，必须传入） | $\theta$ | **待优化的模型参数**。通常直接传 `model.parameters()`，告诉优化器要更新哪些张量。 | （无显式校验，由 PyTorch 基类处理） |
| `lr` | `float` | `1e-3` | $\eta$ | **学习率（基础步长）**。控制参数更新的基础幅度，是优化器中最敏感的超参数之一。值太大容易震荡不收敛，太小收敛太慢。 | 必须 $\geq 0.0$ |
| `betas` | `tuple[float, float]` | `(0.9, 0.999)` | $(\beta_1, \beta_2)$ | **一阶矩/二阶矩的衰减系数**。<br> - $\beta_1$ (第0位)：控制 Momentum 惯性的保留比例（常用0.9，意为“保留上一步90%的速度”）。<br> - $\beta_2$ (第1位)：控制 RMSProp 梯度平方的累积比例（常用0.999，对历史梯度的记忆更长）。 | 两个值都必须在 $[0.0, 1.0)$ 之间 |
| `eps` | `float` | `1e-8` | $\epsilon$ | **数值稳定极小值**。加在 Adam 更新公式的分母 $\sqrt{\hat{v}_t} + \epsilon$ 中，防止分母为 0 导致计算出 `inf` 或 `nan`。 | 必须 $\geq 0.0$ |
| `weight_decay` | `float` | `0.01` | $\lambda$ | **权重衰减系数（L2正则解耦版）**。AdamW 的核心参数，控制独立于梯度更新的参数收缩力度，每一步都会把参数往 0 拉一点，防止过拟合。 | 必须 $\geq 0.0$ |

In [3]:
import math
from torch.optim import Optimizer

class AdamW(Optimizer):
    def __init__(
            self,
            params,
            lr: float=1e-3,
            betas: tuple[float, float]=(0.9, 0.999),        # (β1, β2)：一阶矩/二阶矩的惯性系数
            eps: float=1e-8,
            weight_decay: float=0.01,
    ):
        if lr < 0.0:
            raise ValueError(f"Invalid learning rate: {lr}")
        if eps < 0.0:
            raise ValueError(f"Invalid epsilon: {eps}")
        if not 0.0 <= betas[0] < 1.0:
            raise ValueError(f"Invalid beta1: {betas[0]}")
        if not 0.0 <= betas[1] < 1.0:
            raise ValueError(f"Invalid beta2: {betas[1]}")
        if weight_decay < 0.0:
            raise ValueError(f"Invalid weight_decay: {weight_decay}")
        
        defaults = dict(
            lr=lr,
            betas=betas,
            eps=eps,
            weight_decay=weight_decay,
        )
        super().__init__(params, defaults)

    @torch.no_grad()        # 优化器 step 不需要计算梯度，禁用 autograd 节省内存
    def step(self):
        # 遍历所有参数组（对应后面 build_optimizer 里的 decay/no_decay 分组）
        for group in self.param_groups:
            lr = group["lr"]
            beta1, beta2 = group["betas"]
            eps = group["eps"]
            wd = group["weight_decay"]

            for p in group["params"]:
                if p.grad is None:
                    continue

                grad = p.grad
                if grad.is_sparse:
                    raise RuntimeError("AdamW 不支持稀疏梯度")
                
                # 获取该参数的「状态字典」（用来存 m_t, v_t, step 这些历史信息）
                state = self.state[p]

                # 状态初始化（只在第一次step的时候执行）
                if len(state) == 0:
                    state["step"] = 0   # 迭代步数t初始为0
                    # 一阶矩m_t（动量的惯性速度），初始化为和参数形状相同的0
                    state["exp_avg"] = torch.zeros_like(p, memory_format=torch.preserve_format)
                    # 二阶矩v_t（RMSProp的梯度平方累积）
                    state["exp_avg_sq"] = torch.zeros_like(p, memory_format=torch.preserve_format)

                exp_avg = state["exp_avg"]          # 取出 m_{t-1}
                exp_avg_sq = state["exp_avg_sq"]    # 取出 v_{t-1}

                state["step"] = state["step"] + 1
                t = state["step"]

                # 核心 AdamW 更新逻辑
                # 1）更新一阶矩（继承 Momentum：管方向和惯性）
                # m_t = β1 * m_{t-1} + (1 - β1) * g_t
                exp_avg.mul_(beta1).add_(grad, alpha=1-beta1)   # 原地操作_  节省内存

                # 2）更新二阶矩（继承 RMSProp，管自适应步长）
                # v_t = β2 * v_{t-1} + (1-β2) * g_t^2
                # addcmul_是逐元素相乘再相加
                exp_avg_sq.mul_(beta2).addcmul_(grad, grad, value=1-beta2)

                # 3）偏差修正（解决初始值为0造成的前几步偏差）
                bias_correction1 = 1-beta1**t
                bias_correction2 = 1-beta2**t

                # 公式：m̂_t = m_t / (1 - β1^t),  v̂_t = v_t / (1 - β2^t)
                m_hat = exp_avg / bias_correction1
                v_hat = exp_avg_sq / bias_correction2

                # 4）解耦的权重衰减
                # θ_t = θ_t * (1 - η * λ)
                if wd != 0:
                    p.mul_(1-lr*wd)

                # 5）最终Adam梯度更新
                # 公式：denom = √(v̂_t) + ϵ
                denom = v_hat.sqrt().add_(eps)
                # 公式：θ_{t+1} = θ_t - η * (m̂_t / denom)
                # addcdiv_是逐元素相除再相加
                p.addcdiv_(m_hat, denom, value=-lr)

def build_optimizer(model: torch.nn.Module, lr: float, weight_decay: float):
    """
    不是所有参数都需要权重衰减
    """
    decay_params = []
    no_decay_params = []

    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue    # 冻结的参数，不需要更新梯度

        # --- 经验规则判断 ---
        # 以下几类参数通常不做 weight decay：
        # 1. p.ndim == 1：1D 参数（比如 LayerNorm 的 weight/bias、某些偏置项）
        # 2. name.endswith(".bias")：所有的 bias 项
        # 3. "norm" / "ln" in name.lower()：所有归一化层的参数（LayerNorm, RMSNorm 等）
        if p.ndim == 1 or name.endswith(".bias") or "norm" in name.lower() or "ln" in name.lower():
            no_decay_params.append(p)
        else:
            decay_params.append(p)

    # 构建 PyTorch Optimizer 需要的 param_groups 结构
    # 两组参数用不同的 weight_decay，但共享 lr, betas 等其他超参数
    param_groups = [
        {"params": decay_params, "weight_decay": weight_decay},
        {"params": no_decay_params, "weight_decay": 0.0},
    ]

    return AdamW(param_groups, lr=lr)

AdamW 每一步都在：
1. 更新一阶矩 `m`
2. 更新二阶矩 `v`
3. 做偏差修正
4. 算出 Adam 的更新步长
5. 独立做 weight decay
6. 修改参数

## 5.3 学习率调度
> 起步时不能一脚油门到底，逼近终点时也不能还在林飙，中间才是可以高速巡航的阶段
如果从头到尾都用同一个 lr，通常会碰到两个问题：
- 训练初期太猛
    - 刚初始化的模型很脆，梯度也很乱。
    - 如果一上来就用最大 lr，很容易把 loss 直接打爆。
- 训练后期太粗
    - 快收敛时，如果 lr 还很大，优化器会在低谷附近来回横跳，难以下到更精细的位置。

所以现代训练常见节奏就是 **warmup → decay**
### 5.3.1 Warmup
让模型在训练最不稳定的时候，先用小步子试探。
常见做法是线性 warmup：
$$
\alpha_t = \alpha_{\max}\cdot\frac{t}{T_{\text{warmup}}}
$$
- 第 0 步：$\alpha_0 = 0$（车停着，完全不动）
- 第 $T_{\text{warmup}}/2$ 步：$\alpha_t = \alpha_{\text{max}}/2$（速度加到一半）
- 第 $T_{\text{warmup}}$ 步：$\alpha_t = \alpha_{\text{max}}$（速度加到峰值，Warmup 结束）
### 5.3.2 Cosine Decay
常用余弦衰减的原因是它平滑。
相比那种突然砍半的 step decay，cosine 的好处是：
- **没有突兀跳变**：余弦曲线是平滑的，学习率慢慢降，训练更稳
- **后期更柔和**：快到 $\alpha_{\text{min}}$ 时，下降速度越来越慢，给模型足够时间微调
- **适合长时间训练**：LLM 通常训练几万到几十万步，Cosine Decay 能完美覆盖这个周期

$$\alpha_t = \alpha_{\text{min}} + \frac{1}{2}(1 + \cos(\pi r))(\alpha_{\text{max}} - \alpha_{\text{min}})$$
- $r$：Decay 阶段的**归一化进度**，$r = \frac{t - T_{\text{warmup}}}{T_{\text{cosine}} - T_{\text{warmup}}}$（从 0 变到 1）
- $\alpha_{\text{min}}$：最小学习率（比如 $\alpha_{\text{max}}$ 的 1/10 或 1/100）

- 当 $r=0$（刚进入 Decay 阶段）：$\cos(0) = 1$，所以 $\alpha_t = \alpha_{\text{min}} + \frac{1}{2}(1+1)(\alpha_{\text{max}}-\alpha_{\text{min}}) = \alpha_{\text{max}}$（保持峰值）
- 当 $r=0.5$（Decay 中期）：$\cos(\pi/2) = 0$，所以 $\alpha_t = \alpha_{\text{min}} + \frac{1}{2}(1+0)(\alpha_{\text{max}}-\alpha_{\text{min}}) = \frac{\alpha_{\text{max}}+\alpha_{\text{min}}}{2}$（降到一半）
- 当 $r=1$（Decay 结束）：$\cos(\pi) = -1$，所以 $\alpha_t = \alpha_{\text{min}} + \frac{1}{2}(1-1)(\alpha_{\text{max}}-\alpha_{\text{min}}) = \alpha_{\text{min}}$（降到最小）
### 5.3.3 学习率三阶段
1. **Warmup**：从 0 线性爬到 `max_lr`
2. **Cosine decay**：从 `max_lr` 平滑降到 `min_lr`
3. **After schedule**：停在 `min_lr`
> scheduler 通常是按 step，不是按 epoch

In [4]:
def get_lr_cosine_schedule(
        it: int,                # 当前 step
        max_lr: float,
        min_lr: float,
        warmup_iters: int,      # warmup 总步数
        cosine_cycle_iters: int,# 余弦衰减结束步数
) -> float:
    # 1）warmup
    if it < warmup_iters:
        return max_lr * it/warmup_iters
    
    # 2）after decay
    if it >= cosine_cycle_iters:
        return min_lr
    
    # 3）cosine decay
    # 1. 计算归一化进度r
    decay_ratio = (it - warmup_iters) / (cosine_cycle_iters - warmup_iters)

    # 2. 计算余弦系数coeff
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    
    # 3. 计算当前LR
    # α_min + coeff * (α_max - α_min)
    return min_lr + coeff * (max_lr - min_lr)

## 5.4 梯度裁剪
即便有了 scheduler，训练时还是可能突然出现一次 **极大的梯度（梯度爆炸）**。
这个时候，一般不允许优化器照单全收。

1. forward
2. loss
3. backward
4. **clip gradients**
5. optimizer.step()

这里可以看到，梯度裁剪裁剪的对象不是参数而是梯度，发生在 **反向传播之后、参数更新之前**。

### 5.4.1 全局 L2 范数
| 维度 | L1范数 | L2范数 |
|------|--------|--------|
| 计算公式 | 所有元素的绝对值之和 | 所有元素的平方和开根号 |
| 别名 | 曼哈顿范数、出租车范数 | 欧几里得范数、欧式距离 |
| 对异常值 | 不敏感，鲁棒性强 | 敏感，平方会放大极端值的影响 |
| 优化特性 | 产生稀疏解（很多元素变0） | 产生平滑小值解（元素都很小，非0） |
| 数学性质 | 0点不可导，优化难度高 | 处处平滑可导，完美适配梯度下降 |
| 用途 | 极少用，主要用于稀疏化场景 | 梯度裁剪、权重衰减、Adam二阶矩，核心标配 |

把**所有参数的梯度**看成一个大向量，计算它的 L2 范数：
$$\|g\|_2 = \sqrt{\sum_{p \in \theta} \|g_p\|_2^2}$$
- $g_p$：第 $p$ 个参数的梯度
- $\|g\|_2$：所有梯度的全局总范数

### 5.4.2 裁剪规则
如果 $\|g\|_2 > M$（$M$ 是 `max_norm`，比如 1.0，深度学习社区经过大量实验验证的安全值），就做**等比例缩放**：
$$g \leftarrow g \cdot \frac{M}{\|g\|_2 + \epsilon}$$
> 如果训练中发现梯度经常被裁（Log 里 `total_norm` 经常大于 1.0），可以稍微调大一点（比如 1.5 或 2.0）。如果训练还是不稳定，再调小一点（比如 0.5）。

全局而不是逐层的好处在于：
1.  **保留所有梯度的相对方向关系**：
    *   比如参数 A 的梯度是 3，参数 B 的梯度是 6（比例 1:2）。
    *   全局裁剪后，它们变成 1 和 2（比例还是 1:2），相对关系完全没变。
2.  **不会出现割裂感**：
    *   如果逐层裁剪，可能出现“某层梯度被裁得只剩 0.1，某层完全不动”的情况。
    *   全局裁剪是“一视同仁”，所有梯度一起按比例缩，团队方向一致。

In [5]:
from collections.abc import Iterable

def clip_grad_norm(
        parameters: Iterable[torch.nn.Parameter],
        max_norm: float,                                # 梯度最大允许范数 M（比如 1.0）
        eps: float=1e-6,
) -> float:
    # 1）过滤析出有梯度的参数
    params_with_grad = [p for p in parameters if p.grad is not None]
    if not params_with_grad:
        return 0.0      # 你这参数传过来怎么没梯度

    # 2）计算全局梯度的L2范数
    total_norm_sq = 0.0
    for p in params_with_grad:
        # detach是为了不把这个计算放入计算图，节省内存
        param_norm = torch.norm(p.grad.detach(), p=2)
        # item把 0 维 Tensor 转换成 Python 原生 float
        total_norm_sq = total_norm_sq + param_norm.item()**2

    total_norm = total_norm_sq**0.5

    # 3）等比例裁剪
    if total_norm > max_norm:
        clip_coef = max_norm / (total_norm + eps)
        # 对所有有梯度的参数，其梯度乘以裁剪系数
        for p in params_with_grad:
            p.grad.detach().mul_(clip_coef)

    return total_norm   # 方便记录和debug

## 5.5 数据加载与批处理（Data Loader）
语言模型训练跟很多图像任务不一样，它通常面对的是一条很长很长的 token 流。
而我们要做的，是从这条 token 流里不断切出很多训练样本。
对于 language model，batch 里的 x 和 y 到底是什么？

假设 token 流是`[t0, t1, t2, t3, t4, t5, ...]`；
如果上下文长度 L=4，从位置 i=0 切一段：
- 输入 x：`[t0, t1, t2, t3]`
- 输出 y：`[t1, t2, t3, t4]`

所以一个样本本质上是 **原序列的一段窗口，以及它向右平移一位的版本**。
你不是只拿 `[t0, t1, t2, t3]` 去预测 `t4`，
而是让模型在序列的每个位置都做一次 next-token prediction：
* `t0 -> t1`
* `t1 -> t2`
* `t2 -> t3`
* `t3 -> t4`
所以一个长度为 `L` 的窗口，会提供 `L` 个 token-level 监督信号。
> 这也是 Transformer 训练效率高的原因之一。


In [6]:
import numpy as np
import numpy.typing as npt

def get_batch(
        dataset: npt.NDArray,
        batch_size: int,
        max_seq_length: int,
        device: str,
) -> tuple[torch.Tensor, torch.Tensor]:     # 返回x: [B, S] y:[B, S]
    n = len(dataset)
    # 输入要取到 i : i + L，输出要取到 i + 1 : i + L + 1
    # 因此最大的合法起点必须保证 i + L + 1 <= n
    max_idx = n - max_seq_length - 1
    if max_idx < 0:
        raise ValueError("Dataset is shorted than max_seq_len + 1")
    
    # 1）随机选择batch的起点
    ix = torch.randint(0, max_idx+1, (batch_size,)) # 形状 [batch_size]，每个元素是 0 到 max_idx 之间的随机整数
    # 因为 randint 是「左闭右开」，所以 high 要 +1 才能取到 max_idx

    # 2）切出输入x和目标y
    # 切 x：对每个起始位置 i，取 dataset[i : i+max_seq_length]
    x = torch.stack([
        torch.from_numpy(dataset[i : i + max_seq_length].astype(np.int64))# 把 numpy 数组转成 torch 张量
            # token 索引必须是 int64（LongTensor），因为 PyTorch Embedding 层默认只接受 LongTensor
        for i in ix
    ])      # 把 batch_size 个 [S] 的张量堆成 [B, S] 的张量

    # 切 y：对每个起始位置 i，取 dataset[i+1 : i+max_seq_length+1]
    y = torch.stack([
        torch.from_numpy(dataset[i+1 : i+max_seq_length+1].astype(np.int64))
        for i in ix
    ])

    return x.to(device), y.to(device)

> BTW，数据通常会存成 `.bin + memmap`。如果你把预处理后的 token 文件整个读进 RAM，内存会炸。
> `np.memmap`不要求你把整个数据一次性读进内存，而是像“虚拟地映射”在那，等你切片时才按需加载
> 还有一些prefetch的操作，这里不作深入

## 5.6 模型保存与恢复
想断点续训，只存模型参数远远不够。
因为训练不仅仅是一个静态模型，它还有很多动态状态：
- 模型参数
- 优化器状态
  * 一阶矩 `m`
  * 二阶矩 `v`
  * 每个参数的 step 计数
- 迭代步数 iteration

在 PyTorch 里：
* `model.state_dict()`：保存参数和 buffer
* `optimizer.state_dict()`：保存优化器内部状态
所以最稳妥的 checkpoint 方案，就是把它们统一打包成一个字典。

In [7]:
import os
import typing

def save_checkpoint(
        model: torch.nn.Module,
        optimizer: torch.optim.Optimizer,
        iteration: int,
        out: typing.Union[str, os.PathLike]
):
    checkpoint = {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "iteration": iteration,
    }
    torch.save(checkpoint, out)

def load_checkpoint(
        src: typing.Union[str, os.PathLike],
        model: torch.nn.Module,
        optimizer: torch.optim.Optimizer,
) -> int:
    checkpoint = torch.load(src, map_location="cpu")
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    return checkpoint["iteration"]

如果模型已经搬到 GPU，optimizer state 也要跟着上去。
先在 CPU 上 load checkpoint，再把 model 移到 GPU，固然模型参数会去 GPU，但 optimizer 里面缓存的一些状态张量可能还留在 CPU。
最稳妥的方式通常是：
* 先构造模型并 `.to(device)`
* 再加载 optimizer state
* 必要时把 optimizer state tensor 手动搬运到目标 device

## 5.7 端到端训练流程实现
**一轮训练 step 的顺序**：

1. 根据当前 step 计算 lr
2. 取一个 batch：`x, y`
3. `logits = model(x)`
4. `loss = cross_entropy(logits, y)`
5. `optimizer.zero_grad()`
6. `loss.backward()`
7. `clip_gradient_norm(...)`
8. `optimizer.step()`
9. 记录日志 / 跑验证 / 保存 checkpoint

**lr → batch → forward → loss → backward → clip → step → eval/log/save**

| 训练循环步骤 | 代码 | 下山类比 | 作用 |
|--------------|------|----------|------|
| 1. 调学习率 | `lr = get_lr_cosine_schedule(...)` | 决定“这一步迈多大” | 控制整体步长节奏 |
| 2. 取数据 | `x, y = get_batch(...)` | 看一眼“当前周围的路况” | 获取训练样本 |
| 3. 前向传播 | `logits = model(x)` | 站在当前位置，“猜一下下一步往哪走” | 模型根据输入做预测 |
| 4. 算损失 | `loss = cross_entropy(logits, y)` | 看一下“猜得对不对，离谷底还有多远” | 计算预测误差 |
| 5. 反向传播 | `loss.backward()` | 拿出指南针，**算出“往哪个方向走，海拔下降最快”** | 计算梯度 $g_t$（方向） |
| 6. 梯度裁剪 | `clip_gradient_norm(...)` | 如果指南针指的方向“拉力太猛”，就把拉力按比例减小 | 防止梯度爆炸 |
| 7. **Step（核心）** | `optimizer.step()` | **真正迈一步！** 按照算好的方向和步长，移动到新位置 | **更新模型参数 $\theta$** |
| 8. 评估/记录/保存 | `evaluate(...)` / `save_checkpoint(...)` | 拍个照，记录一下现在的位置，休息一下存个档 | 监控训练进度 |

In [11]:
import sys
from pathlib import Path

ROOT = Path("..").absolute()
sys.path.append(str(ROOT))

import import_ipynb
import os
import numpy as np
import torch
from typing import Optional

from decoder.decoder import TransformerLM

### 5.7.1 混合精度训练AMP
前人写训练循环，默认都是 `float32`。但在真正训练大模型时，它通常太贵了：显存更大、吞吐更低。于是现代训练会尽量把一部分算子放到更低精度去跑，同时保留关键位置的数值稳定性。

AMP 不是“全模型都降精度”，而是“让大部分安全的地方用低精度跑，把真正脆弱的地方留给更稳的精度”。

### 5.7.2 梯度累积
很多时候我想要更大的有效 batch size，但单卡显存装不下。梯度累积的思路就是：用多个小 batch 连续做 forward/backward，把梯度先攒起来，等攒够了再 `step()` 一次。

因为训练里真正重要的，不是“每次显存里塞多少样本”，而是 **每次参数更新，到底累积了多少样本的梯度信息。**

所以如果我每 4 个 micro-batch 才更新一次参数，那么：
$$
\text{effective batch size} = \text{micro batch size} \times 4
$$
这在单卡或小机器上特别有价值。
> 因为我想模拟的是“一个更大 batch 的平均梯度”，所以每个 micro-batch 的 loss 要先缩小一份，再累计 backward。因此 loss 要先除以 `grad_accum_steps`
### 5.7.3 标签平滑
前面我们写的是标准 one-hot 交叉熵：正确标签就是 1，其他标签都是 0。
但这会带来一个副作用：模型可能越来越“自信过头”。

标签平滑思路则是，别把正确类写成绝对 1，而是留一点点概率质量给其他类。原始 Transformer 论文就在训练里使用了 label smoothing，(\epsilon_{ls}=0.1)；

它会让 perplexity（衡量自回归语言模型「预测下一个 token 的确定性」的核心指标。PPL 越低，说明模型对自己的预测越 “有把握”，越确定下一个词该出什么；反之 PPL 越高，模型越 “犹豫”。）变差一点，但能提升准确率和 BLEU（双语评估替补值，机器翻译、文本生成任务的通用打分工具，专门衡量模型生成的内容和人工标准答案的匹配度，分数 0-100，越高说明生成内容越准、越通顺。原始 Transformer 论文就是做机器翻译的，所以用 BLEU 衡量效果）。

这对 seq2seq、分类或一些有同义替换空间的任务尤其自然。
不过对纯 causal LM 预训练来说，基本都没有使用。标签平滑的核心作用是正则化、防止过拟合，但大模型预训练用的是万亿级别的海量语料，模型本身很难过拟合，标签平滑带来的收益极小。并且标签平滑会让模型收敛变慢，还会升高 PPL，对预训练的效率和效果都有负面影响。

In [ ]:
@torch.no_grad()
def evaluate(model, dataset, batch_size, max_seq_length, device, eval_iters=20):
    model.eval()
    losses = []

    for _ in range(eval_iters):
        x, y = get_batch(dataset, batch_size, max_seq_length, device)
        logits = model(x)
        loss = cross_entropy(logits, y)
        losses.append(loss)  # 这里先存 tensor，不用 .item()

    model.train()
    # 用 torch.stack 把 list 转成 tensor，再 .mean()
    return torch.stack(losses).mean().item()

In [ ]:
def train(
    # --------------------------
    # 必需参数（数据路径）
    # --------------------------
    train_data_path: str,
    valid_data_path: str,
    # --------------------------
    # 模型超参数
    # --------------------------
    batch_size: int = 32,
    context_length: int = 256,
    d_model: int = 512,         # 模型隐藏层维度
    num_layers: int = 4,
    num_heads: int = 8,
    d_ff: int = 1536,           # 前馈网络中间层维度
    vocab_size: int = 10000,
    rope_theta: float = 10000.0,
    # --------------------------
    # 训练超参数
    # --------------------------
    lr: float = 6e-4,
    min_lr: float = 6e-5,
    warmup_iters: int = 1000,
    max_iters: int = 10000,
    weight_decay: float = 0.1,
    max_norm: float = 1.0,
    # --------------------------
    # 路径与设备配置
    # --------------------------
    out_dir: str = "out",
    device: Optional[str] = None,
    dtype: torch.dtype = torch.float32,
    # --------------------------
    # AMP 配置
    # --------------------------
    use_amp: Optional[bool] = None,  # 是否启用 AMP，默认自动判断
    amp_dtype: torch.dtype = torch.float16,  # AMP 计算精度（FP16 或 BF16）
    # --------------------------
    # 梯度累积配置
    # --------------------------
    grad_accum_steps: int = 1,  # 梯度累计步数，1 表示不累计
) -> None:
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    os.makedirs(out_dir, exist_ok=True)

    # 1）加载数据
    train_data = np.memmap(train_data_path, dtype=np.uint16, mode="r")
    val_data = np.memmap(valid_data_path, dtype=np.uint16, mode="r")

    # 2）初始化模型
    model = TransformerLM(
        vocab_size=vocab_size,
        max_seq_len=context_length,
        d_model=d_model,
        num_layers=num_layers,
        num_heads=num_heads,
        d_ff=d_ff,
        rope_theta=rope_theta,
        device=device,
        dtype=dtype,
    ).to(device)

    # 3）初始化优化器
    optimizer = build_optimizer(
        model=model,
        lr=lr,
        weight_decay=weight_decay
    )

    # 4）恢复检查点（如果有的话）
    ckpt_path = os.path.join(out_dir, "ckpt.pt")
    start_iter = 0
    if os.path.exists(ckpt_path):
        start_iter = load_checkpoint(ckpt_path, model, optimizer)
        print(f"Resumed training from step {start_iter}")

    # AMP
    if use_amp is None:
        # 自动判断：只有在 CUDA 设备上才启用 AMP
        use_amp = device.startswith("cuda")
    # 初始化 GradScaler（仅在 CUDA 上有效，用于防止 FP16 下溢）
    # 注意：PyTorch 2.0+ 推荐用 torch.amp.GradScaler
    scaler = torch.amp.GradScaler(device, enabled=use_amp)
    
    # 清零梯度移到循环最外面（只在开始时清一次），因为要适配梯度累积
    optimizer.zero_grad(set_to_none=True)
    
    # 5）主训练循环
    for it in range(start_iter, max_iters):
        # 5.1）学习率调度
        current_lr = get_lr_cosine_schedule(
            it=it,
            max_lr=lr,
            min_lr=min_lr,
            warmup_iters=warmup_iters,
            cosine_cycle_iters=max_iters,
        )
        # 手动更新优化器的学习率
        for param_group in optimizer.param_groups:
            param_group["lr"] = current_lr

        # 5.2）采样batch
        x, y = get_batch(
            train_data,
            batch_size=batch_size,
            max_seq_length=context_length,
            device=device,
        )

        # 5.3）清零梯度（没有梯度累积的话正常是放这里）
        # optimizer.zero_grad(set_to_none=True)   # 这样比把梯度全置零更省一点内存写入

        # 5.4）前向传播计算损失(用 autocast 上下文管理器了!)
        # logits = model(x)
        # loss = cross_entropy(logits, y)
        with torch.autocast(device_type=device, dtype=amp_dtype, enabled=use_amp):
            # 在这个上下文里，PyTorch 会自动把合适的运算转到 FP16/BF16
            logits = model(x)
            loss = cross_entropy(logits, y)
            # 因为梯度要累计 grad_accum_steps 次，所以每次的 loss 要相应缩小，
            # 保证最终累计的梯度量级和用单步大 Batch 一致
            loss = loss / grad_accum_steps

        # 5.5）反向传播
        # scaler.scale() 会把 loss 放大，防止 FP16 梯度下溢
        scaler.scale(loss).backward()   # 梯度会自动累加
        
        # 5.6）梯度裁剪
        # 如果不先 unscale，梯度是被放大过的，裁剪阈值会失效
        if (it + 1) % grad_accum_steps == 0:
            scaler.unscale_(optimizer)
            grad_norm = clip_grad_norm(model.parameters(), max_norm)

        # 5.7）优化器更新参数
        # scaler.step()会先检查梯度是否有inf/nan，没问题再UNscale并更新
            scaler.step(optimizer)
            scaler.update()
            # 更新完参数后清零梯度
            optimizer.zero_grad(set_to_none=True)
        else:   # 没累计够步数时，grad_norm 设为 None 或 0，避免日志报错
            grad_norm = 0.0

        # 5.8）日志记录与验证集评估
        if it % 100 == 0 or it == max_iters - 1:
            # 在验证集上评估
            val_loss = evaluate(
                model,
                val_data,
                batch_size=batch_size,
                max_seq_length=context_length,
                device=device,
                eval_iters=20,
            )
            # 计算有效步数（等效大 Batch 步数）
            effective_step = (it + 1) // grad_accum_steps
            print(
                f"iter {it:06d} (eff: {effective_step:06d}) | "
                f"train_loss {loss.item() * grad_accum_steps:.4f} | "  # 日志里把 loss 乘回去，显示真实 loss
                f"val_loss {val_loss:.4f} | "
                f"lr {current_lr:.2e} | "
                f"grad_norm {grad_norm:.2f}"
            )

        # 5.8）保存检查点
        if it % 1000 == 0 and it > 0:
            save_checkpoint(model, optimizer, it, ckpt_path)

    final_ckpt_path = os.path.join(out_dir, "ckpt_final.pt")
    save_checkpoint(model, optimizer, max_iters, final_ckpt_path)
    print(f"Training finished! Final checkpoint saved to {final_ckpt_path}")

### 5.7.4 余弦退火带重启
技术出自 **Stochastic Gradient Descent with Warm Restarts**（带热重启的随机梯度下降）。

在之前的训练流程里，学习率是这样的：
1.  **Warmup**：从 0 线性爬到 `max_lr`
2.  **Cosine Decay**：从 `max_lr` **一次平滑下降**到 `min_lr`，之后就一直保持 `min_lr` 不动

而 SGDR 的学习率曲线是**周期性**的：
1.  **第一个周期**：从 `max_lr` 平滑下降到 `min_lr`（和普通 Cosine Decay 一样）
2.  **重启**：突然「跳回」到 `max_lr`（给优化器重新注入活力）
3.  **第二个周期**：再次从 `max_lr` 下降到 `min_lr`，但**周期长度可能变长**（比如是第一个周期的 2 倍）
4.  **循环往复**：不断「下降 → 重启 → 下降」

普通 Cosine Decay 有个潜在问题：**可能过早陷入一个「不太好的局部最优」**。
- 比如你滑到了一个小盆地（局部最小值），但其实旁边不远处有一个更深、更好的大峡谷（全局最优）。
- 但普通 Cosine Decay 学习率已经降到很低了，没有力气跳出这个小盆地，只能困在里面。
- 我周期性地把学习率拉回来（重启），给优化器一点「探索活力」，让它有机会跳出当前的小盆地，去寻找更好的最小值。

In [ ]:
def get_lr_cosine_restarts(
    it: int,                          # 当前步数
    max_learning_rate: float,         # 峰值学习率（重启后的起点）
    min_learning_rate: float,         # 最小学习率（每个周期的终点）
    first_cycle_iters: int,           # 第一个周期的长度（比如 1000 步）
    cycle_mult: int = 2,              # 周期长度的倍数（比如 2 表示每个周期是前一个的 2 倍）
) -> float:
    """
    SGDR 风格：余弦退火 + 周期重启
    """
    # ------------------------------------------------------
    # 1. 初始化：第一个周期的长度
    # ------------------------------------------------------
    cycle_len = first_cycle_iters  # 当前周期的长度，初始是第一个周期
    cur = it                        # 剩余步数，用来找当前 it 属于哪个周期

    # ------------------------------------------------------
    # 2. 循环：找到当前 it 落在第几个周期里
    # ------------------------------------------------------
    # 逻辑：如果剩余步数 cur >= 当前周期长度 cycle_len，说明 it 不在这个周期
    #       那就减去这个周期的长度，把周期长度乘以 cycle_mult，继续判断
    while cur >= cycle_len:
        cur -= cycle_len          # 减去当前周期的长度，剩余步数进入下一个周期
        cycle_len *= cycle_mult   # 下一个周期的长度变长（比如 *2）

    # ------------------------------------------------------
    # 3. 计算当前周期内的归一化进度
    # ------------------------------------------------------
    # 此时 cur 是「当前周期内的步数」（从 0 到 cycle_len-1）
    # ratio 是当前周期内的进度（从 0.0 到 1.0）
    ratio = cur / cycle_len

    # ------------------------------------------------------
    # 4. 用余弦公式计算当前学习率
    # ------------------------------------------------------
    # 和普通 Cosine Decay 一样：
    # ratio=0.0（周期起点）→ coeff=1.0 → lr=max_lr
    # ratio=1.0（周期终点）→ coeff=0.0 → lr=min_lr
    coeff = 0.5 * (1.0 + math.cos(math.pi * ratio))
    return min_learning_rate + coeff * (max_learning_rate - min_learning_rate)

### 5.7.5 分布式训练 ZeRO/FSDP
真实大模型训练不可能一直是“整模型复制到每张卡”。如果我用普通数据并行，每张卡都要持有一整份：
* 参数
* 梯度
* 优化器状态
ZeRO 和 FSDP 本质上都在解决这件事：**把模型状态分片，而不是全量复制。**
DeepSpeed 的 ZeRO 把优化器状态、梯度和参数分区到不同 data-parallel worker 上，Stage 1/2/3 分别对应“只切 optimizer state / 再切 gradients / 再切 parameters”。

PyTorch 的 `fully_shard` 文档则直接把 FSDP 定义成：对模块参数、梯度和优化器状态进行 fully sharded data parallel 以节省内存。

* **ZeRO**：DeepSpeed 语境下非常经典的分片训练路线，强调按 stage 逐步切更多状态。
* **FSDP**：PyTorch 官方体系下的全分片并行方案，今天已经是非常主流的官方路线；文档里甚至已经把 FSDP2 当作新前端去讲。

| 方案     | 核心想法              | 备注       |
| ------ | ----------------- | ------------- |
| ZeRO-1 | 切 optimizer state | 先减最肥的那部分      |
| ZeRO-2 | 再切 gradients      | 继续减内存         |
| ZeRO-3 | 连 parameters 也切   | 真正大幅省显存       |
| FSDP   | 参数/梯度/优化器状态统一分片   | PyTorch 官方主路线 |
